# Module 1: Helm - Installing LangSmith

## Overview

This notebook walks through installing LangSmith using the **official `langchain-ai/helm` chart**.

### Key Principles

- ✅ Use the **official** Helm chart (do not fork)
- ✅ Pin chart versions for reproducibility
- ✅ Create minimal, sane values file
- ✅ Inject required secrets properly
- ✅ Render templates before install
- ✅ Understand that "helm install succeeded" ≠ "system is healthy"

### What We'll Install

- LangSmith application components
- External service connections (RDS, Redis, S3)
- Resource requests & limits
- Ingress configuration

**Estimated time:** 45-60 minutes


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## Understanding the Official Helm Chart

The `langchain-ai/helm` repository contains the official LangSmith Helm chart. We use the **official** chart because:

1. **Support:** Support will expect standard configurations
2. **Updates:** Official charts receive security and feature updates
3. **Documentation:** Official charts are documented and tested
4. **Compatibility:** Ensures compatibility with Terraform outputs

**Important:** We do **not** fork the upstream repository. We reference it directly.


In [ ]:
import os
import json
import re
from pathlib import Path
from shared._validation import require_env, ok, warn, fail
from shared._shell import run

def expand_env_vars(path_str: str) -> str:
    """Expand environment variable references in a path string."""
    # Expand $VAR and ${VAR} references
    def replace_var(match):
        var_name = match.group(1) or match.group(2)
        return os.environ.get(var_name, match.group(0))
    
    # Replace $VAR and ${VAR} patterns
    path_str = re.sub(r'\$\{([^}]+)\}|\$([a-zA-Z_][a-zA-Z0-9_]*)', replace_var, path_str)
    return path_str

# Get required configuration
config = require_env(
    "HELM_CHART_REF", "HELM_RELEASE", "HELM_NAMESPACE", 
    "CLUSTER_NAME", "AWS_REGION", "NAMESPACE"
)

# Expand environment variables in the path (e.g., $HELM_REPO_DIR, $HOME)
helm_chart_ref_str = expand_env_vars(config["HELM_CHART_REF"])
helm_chart_ref = Path(helm_chart_ref_str).expanduser().resolve()

helm_release = config["HELM_RELEASE"]
helm_namespace = config["HELM_NAMESPACE"]
cluster_name = config["CLUSTER_NAME"]
region = config["AWS_REGION"]
namespace = config["NAMESPACE"]

print("### Helm Configuration")
print(f"Chart Reference: {helm_chart_ref}")
print(f"Release Name: {helm_release}")
print(f"Namespace: {helm_namespace}")
print(f"Cluster: {cluster_name}")
print(f"Region: {region}\n")

if not helm_chart_ref.exists():
    fail(f"Helm chart path does not exist: {helm_chart_ref}")
    print("\n💡 To fix this:")
    print("   1. Clone the official Helm repository:")
    print("      git clone https://github.com/langchain-ai/helm.git <target-directory>")
    print("   2. Update HELM_CHART_REF in your .env file to point to:")
    print(f"      HELM_CHART_REF=\"<target-directory>/charts/langsmith\"")
    print("   3. Run this notebook again")
    raise RuntimeError(f"Helm chart path not found: {helm_chart_ref}")

ok(f"Helm chart path exists: {helm_chart_ref}")

# Check Helm version
print("\n### Helm Version")
result = run(["helm", "version"], check=True, stream=False)
print(result.stdout)


## Discovering the Chart Path

Verify the Helm chart structure and locate the Chart.yaml file.


In [ ]:
# Verify Helm chart structure
print("### Helm Chart Structure\n")

# Check for Chart.yaml
chart_yaml = helm_chart_ref / "Chart.yaml"
if chart_yaml.exists():
    ok("Found Chart.yaml")
    print("\nChart.yaml contents:")
    print("=" * 60)
    with open(chart_yaml) as f:
        print(f.read())
    print("=" * 60)
else:
    warn("Chart.yaml not found")
    raise RuntimeError(f"❌ Invalid Helm chart: {helm_chart_ref}")

# Check for values.yaml
values_yaml = helm_chart_ref / "values.yaml"
if values_yaml.exists():
    ok("Found values.yaml (default values)")
else:
    warn("values.yaml not found (may be optional)")

# List chart files
print("\n### Chart Files")
chart_files = sorted(helm_chart_ref.glob("*"))
for f in chart_files[:20]:  # Show first 20
    if f.is_file():
        print(f"  📄 {f.name}")
    elif f.is_dir():
        print(f"  📁 {f.name}/")
if len(chart_files) > 20:
    print(f"  ... and {len(chart_files) - 20} more items")


## Pinning Chart Versions

**Critical:** Always pin Helm chart versions for reproducibility. Check the Chart.yaml for the version.


In [ ]:
# Extract chart version
import yaml

with open(chart_yaml) as f:
    chart_info = yaml.safe_load(f)

print("### Chart Version Information\n")
print(f"Chart Name: {chart_info.get('name', 'N/A')}")
print(f"Chart Version: {chart_info.get('version', 'N/A')}")
print(f"App Version: {chart_info.get('appVersion', 'N/A')}")
print(f"Description: {chart_info.get('description', 'N/A')[:100]}...")

ok(f"Using chart version: {chart_info.get('version', 'N/A')}")
print("\n💡 Record this version for reproducibility")


## Loading Terraform Outputs

We need the Terraform outputs from the previous notebook to configure Helm values (RDS, Redis, S3, etc.).


In [ ]:
# Load Terraform outputs
terraform_outputs_file = artifacts_dir / "terraform-outputs.json"

if not terraform_outputs_file.exists():
    warn(f"Terraform outputs file not found: {terraform_outputs_file}")
    print("💡 Run notebook 02_terraform_apply.ipynb first to generate outputs")
    terraform_outputs = {}
else:
    with open(terraform_outputs_file) as f:
        terraform_outputs_raw = json.load(f)
    
    # Unwrap Terraform output format
    terraform_outputs = {}
    for key, value in terraform_outputs_raw.items():
        if isinstance(value, dict) and "value" in value:
            terraform_outputs[key] = value["value"]
        else:
            terraform_outputs[key] = value
    
    ok(f"Loaded Terraform outputs from {terraform_outputs_file}")
    print(f"\nAvailable outputs: {', '.join(terraform_outputs.keys())}")
    
    # Show key outputs (redacted for secrets)
    print("\n### Key Outputs (for reference):")
    for key in ["cluster_name", "rds_endpoint", "redis_endpoint", "s3_bucket"]:
        if key in terraform_outputs:
            val = str(terraform_outputs[key])
            if len(val) > 50:
                print(f"  {key}: {val[:50]}...")
            else:
                print(f"  {key}: {val}")


## Creating a Minimal Values File

We'll create a minimal, sane values file that:
- Connects to external services (RDS, Redis, S3)
- Sets resource requests & limits
- Configures ingress
- Includes required secrets

**Important:** Start minimal. Add complexity only as needed.


In [ ]:
# Check if values file is specified
values_file_env = os.environ.get("VALUES_FILE", "").strip()

if values_file_env:
    values_file_path = Path(values_file_env).expanduser().resolve()
    if values_file_path.exists():
        ok(f"Using values file from environment: {values_file_path}")
        print("💡 Review the values file to ensure it's configured correctly")
    else:
        warn(f"Values file from environment not found: {values_file_path}")
        print("💡 Will need to create a values file")
        values_file_path = None
else:
    values_file_path = None
    print("💡 VALUES_FILE not set in environment")
    print("   We'll create a minimal values file for this deployment")

# If no values file, we'll create one
if not values_file_path:
    values_file_path = artifacts_dir / "langsmith-values.yaml"
    print(f"\nWill create values file at: {values_file_path}")
    print("💡 You can customize this file before installation")


## Injecting Required Secrets

LangSmith requires several secrets:
- **License key** (required)
- Database credentials (if not using IAM auth)
- Redis password (if not using IAM auth)
- S3 credentials (if not using IAM roles)

Let's prepare the secrets.


In [ ]:
# Check for required secrets
print("### Required Secrets\n")

# License key (required)
license_key = os.environ.get("LANGSMITH_LICENSE_KEY", "").strip()
if license_key:
    ok("LANGSMITH_LICENSE_KEY is set")
    print("💡 License key will be used to create Kubernetes secret")
else:
    warn("LANGSMITH_LICENSE_KEY not set")
    print("💡 You must set LANGSMITH_LICENSE_KEY in your .env file")
    print("   Get your license key from LangSmith support")

# Database credentials (may be optional if using IAM auth)
db_user = os.environ.get("DB_USER", "").strip()
db_password = os.environ.get("DB_PASSWORD", "").strip()
if db_user and db_password:
    ok("Database credentials are set")
else:
    print("💡 Database credentials may be optional if using IAM authentication")
    print("   Check your Terraform outputs for connection details")

# Redis password (may be optional if using IAM auth)
redis_password = os.environ.get("REDIS_PASSWORD", "").strip()
if redis_password:
    ok("Redis password is set")
else:
    print("💡 Redis password may be optional if using IAM authentication")

print("\n💡 Secrets will be created as Kubernetes secrets before Helm install")


## Preparing Kubernetes Namespace

Create the namespace if it doesn't exist.


In [ ]:
from shared._k8s_helpers import namespace_exists, kubectl
from shared._aws_helpers import aws_region

# Ensure kubectl is configured
region = aws_region()
run(
    ["aws", "eks", "update-kubeconfig", "--name", cluster_name, "--region", region],
    check=True,
    stream=False
)

# Create namespace if needed
print(f"### Preparing Namespace: {namespace}\n")

if namespace_exists(namespace):
    ok(f"Namespace '{namespace}' already exists")
else:
    print(f"Creating namespace '{namespace}'...")
    kubectl("create", "namespace", namespace, check=True, stream=True)
    ok(f"Namespace '{namespace}' created")


## Creating Kubernetes Secrets

Create the required secrets in the namespace.


In [ ]:
# Create secrets
print("### Creating Kubernetes Secrets\n")

if not license_key:
    raise RuntimeError("❌ LANGSMITH_LICENSE_KEY is required")

# Create license key secret
print("Creating license key secret...")
run(
    [
        "kubectl", "create", "secret", "generic", "langsmith-license",
        f"--from-literal=license-key={license_key}",
        "-n", namespace,
        "--dry-run=client", "-o", "yaml"
    ],
    check=True,
    stream=False
)
# Actually create it (remove dry-run)
run(
    [
        "kubectl", "create", "secret", "generic", "langsmith-license",
        f"--from-literal=license-key={license_key}",
        "-n", namespace
    ],
    check=False,  # May already exist
    stream=True
)
ok("License key secret created/updated")

# Create database secret if credentials provided
if db_user and db_password:
    print("\nCreating database secret...")
    run(
        [
            "kubectl", "create", "secret", "generic", "langsmith-db",
            f"--from-literal=username={db_user}",
            f"--from-literal=password={db_password}",
            "-n", namespace
        ],
        check=False,  # May already exist
        stream=True
    )
    ok("Database secret created/updated")
else:
    print("💡 Skipping database secret (using IAM auth or not needed)")

# Create Redis secret if password provided
if redis_password:
    print("\nCreating Redis secret...")
    run(
        [
            "kubectl", "create", "secret", "generic", "langsmith-redis",
            f"--from-literal=password={redis_password}",
            "-n", namespace
        ],
        check=False,  # May already exist
        stream=True
    )
    ok("Redis secret created/updated")
else:
    print("💡 Skipping Redis secret (using IAM auth or not needed)")

print("\n✅ Secrets preparation complete")


## Rendering Templates Before Install

**Critical:** Always render Helm templates before installing. This lets you:
- Verify the configuration is correct
- Catch errors before deployment
- Review what will be created

This is especially important for understanding resource requests & limits.


In [ ]:
# Render Helm templates
print("### Rendering Helm Templates\n")
print("This shows what Kubernetes resources will be created...\n")

# Use values file if it exists, otherwise use empty values
values_arg = []
if values_file_path and values_file_path.exists():
    values_arg = ["-f", str(values_file_path)]
    print(f"Using values file: {values_file_path}\n")

result = run(
    [
        "helm", "template", helm_release, str(helm_chart_ref),
        "-n", namespace,
        *values_arg,
        "--debug"  # Show computed values
    ],
    check=False,  # Don't fail on warnings
    stream=True
)

# Save rendered templates
rendered_file = artifacts_dir / "helm-rendered-templates.yaml"
with open(rendered_file, "w") as f:
    f.write(result.stdout)
    if result.stderr:
        f.write("\n\nSTDERR:\n")
        f.write(result.stderr)

print(f"\n💡 Rendered templates saved to: {rendered_file}")

if result.returncode == 0:
    ok("Template rendering successful")
    print("\n⚠️  Review the rendered templates above. If they look correct, proceed to install.")
else:
    warn(f"Template rendering had issues (rc={result.returncode})")
    print("💡 Review the errors above before proceeding")


## Installing LangSmith with Helm

**⚠️ WARNING:** This will install LangSmith into your cluster.

Only proceed if:
1. ✅ You've reviewed the rendered templates
2. ✅ Secrets are created
3. ✅ Values file is correct
4. ✅ Terraform outputs are loaded

**Estimated installation time:** 5-10 minutes


In [ ]:
# Install LangSmith with Helm
# ⚠️  UNCOMMENT THE CODE BELOW TO ACTUALLY INSTALL
# This is commented out by default to prevent accidental deployments

print("### Installing LangSmith with Helm\n")
print("⚠️  This cell is currently DISABLED to prevent accidental deployments.\n")
print("To install, uncomment the code below and run this cell.\n")

# UNCOMMENT TO INSTALL:
# print("Installing LangSmith... This may take 5-10 minutes.\n")
# 
# values_arg = []
# if values_file_path and values_file_path.exists():
#     values_arg = ["-f", str(values_file_path)]
# 
# result = run(
#     [
#         "helm", "install", helm_release, str(helm_chart_ref),
#         "-n", namespace,
#         "--create-namespace",
#         *values_arg,
#         "--wait",  # Wait for deployment to be ready
#         "--timeout", "10m"
#     ],
#     check=False,  # Don't fail immediately, we'll check status
#     stream=True
# )
# 
# # Save install output
# install_file = artifacts_dir / "helm-install.txt"
# with open(install_file, "w") as f:
#     f.write(result.stdout)
#     if result.stderr:
#         f.write("\n\nSTDERR:\n")
#         f.write(result.stderr)
# 
# if result.returncode == 0:
#     ok("Helm install completed")
#     print(f"\n💡 Install output saved to: {install_file}")
# else:
#     warn(f"Helm install had issues (rc={result.returncode})")
#     print("💡 Check the output above for errors")

print("💡 To install, edit this cell and uncomment the code above")


## Understanding: "helm install succeeded" ≠ "system is healthy"

**Important:** A successful Helm install only means:
- Resources were created
- Helm release is tracked

It does **not** mean:
- Pods are running
- Services are healthy
- Ingress is working
- Database connections work

We'll validate system health in the next notebook.


In [ ]:
# Check Helm release status
print("### Helm Release Status\n")

result = run(
    ["helm", "list", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)

releases = json.loads(result.stdout)
langsmith_releases = [r for r in releases if r.get("name") == helm_release]

if langsmith_releases:
    release = langsmith_releases[0]
    print(f"Release: {release['name']}")
    print(f"Status: {release['status']}")
    print(f"Chart: {release['chart']}")
    print(f"Namespace: {release['namespace']}")
    print(f"Revision: {release['revision']}")
    
    if release['status'] == 'deployed':
        ok("Helm release is deployed")
        print("\n💡 Remember: 'deployed' doesn't mean healthy!")
        print("   Proceed to validation notebook to check pod status, ingress, etc.")
    else:
        warn(f"Helm release status: {release['status']}")
else:
    warn(f"Helm release '{helm_release}' not found")
    print("💡 If you just installed, wait a moment and check again")


## Summary

### ✅ What We Accomplished

- [ ] Located and verified Helm chart
- [ ] Pinned chart version
- [ ] Loaded Terraform outputs
- [ ] Created/verified values file
- [ ] Created Kubernetes secrets
- [ ] Rendered templates for review
- [ ] Installed LangSmith (if you uncommented the install step)
- [ ] Checked Helm release status

### 📋 Key Takeaways

1. **Use official Helm chart** - Don't fork, reference directly
2. **Pin versions** - Ensures reproducibility
3. **Start minimal** - Add complexity only as needed
4. **Render first** - Always render templates before installing
5. **Secrets matter** - Properly inject required secrets
6. **Install ≠ Healthy** - Validation comes next

### 🎯 Next Steps

Proceed to **04_validate_ingress_and_ui.ipynb** to validate:
- Pod readiness
- PVC binding
- Ingress provisioning
- Endpoint reachability
- Basic UI availability
